# schema-bridge — matcher test

Tests the full pipeline step by step:
1. Parse both model files into Python dicts
2. Extract subgraphs for concepts of interest
3. Run rapidfuzz lexical matching
4. Inspect results

Run cells top to bottom. Each cell depends on the ones above it.

## setup

Add `etl/` folder to the Python path so imports work from the notebook.

In [ ]:
import pathlib

# current directory when jupyter runs — reliable in VS Code notebooks
ROOT = pathlib.Path().resolve()

# if that doesn't point to your project root, hardcode it:
# ROOT = pathlib.Path("C:/Users/........")

GQL_PATH = ROOT / "data_models" / "ilap_graphql_schema.graphql"
TTL_PATH = ROOT / "data_models" / "sdo.ttl"

print("ROOT    :", ROOT)
print("GQL     :", GQL_PATH, "exists:", GQL_PATH.exists())
print("TTL     :", TTL_PATH, "exists:", TTL_PATH.exists())

ROOT    : C:\Users\508719\OneDrive - Aker Solutions\Desktop\ILAP\ilap_sdo_mapping_test
GQL     : C:\Users\508719\OneDrive - Aker Solutions\Desktop\ILAP\ilap_sdo_mapping_test\data_models\ilap_graphql_schema.graphql exists: True
TTL     : C:\Users\508719\OneDrive - Aker Solutions\Desktop\ILAP\ilap_sdo_mapping_test\data_models\sdo.ttl exists: True


## parse both files

In [8]:
from etl.extractor import parse_graphql, parse_sdo

GQL_PATH = ROOT / "data_models" / "ilap_graphql_schema.graphql"
TTL_PATH = ROOT / "data_models" / "sdo.ttl"

graphql_index = parse_graphql(str(GQL_PATH))
sdo_index     = parse_sdo(str(TTL_PATH))

print(f"GraphQL index : {len(graphql_index)} entries")
print(f"SDO index     : {len(sdo_index)} entries")

GraphQL index : 246 entries
SDO index     : 157 entries


### spot-check — look at one entry from each

In [9]:
import pprint

# GraphQL — Activity type (fields truncated for readability)
act = graphql_index["Activity"]
print(f"Activity — kind: {act['kind']}, fields: {len(act['fields'])}")

# show first 5 fields
for name, field in list(act["fields"].items())[:5]:
    print(f"  {name:<30} type={field['type']:<20} is_relation={field['is_relation']}")

Activity — kind: object, fields: 278
  predecessors                   type=Successor            is_relation=True
  resourceAssignments            type=ResourceAssignment   is_relation=True
  resourceAvailabilities         type=ResourceAvailability is_relation=True
  successors                     type=Successor            is_relation=True
  calendar                       type=Calendar             is_relation=True


In [10]:
# SDO — ScheduleActivity class
sa = sdo_index["ScheduleActivity"]
print(f"ScheduleActivity")
print(f"  definition  : {sa['definition'][:100]}")
print(f"  subclass_of : {sa['subclass_of']}")
print(f"  subclasses  : {sa['subclasses']}")
print(f"  properties  : {sa['properties'][:8]} ...")

ScheduleActivity
  definition  : An activity that identifies a piece of work that is required to be undertaken to complete a schedule
  subclass_of : ['Activity']
  subclasses  : ['CancelledActivity', 'AlwaysOnScheduleActivity', 'EarlyFinishActivity', 'EarlyStartActivity', 'LateFinishActivity', 'LateStartActivity']
  properties  : ['actualFinish', 'actualStart', 'earlyFinish', 'earlyStart', 'finishNoEarlierThan', 'finishNoLaterThan', 'freeFloat', 'hasBreakdownStructureElement'] ...


## extract concept subgraphs

Specify your seeds here. The walker finds everything reachable from them.

In [ ]:
from etl.concepts import extract, describe

# change these to scope to different concepts
SEEDS = ["Activity", "Schedule", "Project", "Data", "ScheduleActivity"]

gql_subgraph = extract(graphql_index, SEEDS)
sdo_subgraph = extract(sdo_index,     SEEDS)

print(f"GraphQL subgraph : {len(gql_subgraph)} nodes")
print(f"SDO subgraph     : {len(sdo_subgraph)} nodes")

  [concepts] Warning: seed 'Project' not found in index — skipped
  [concepts] Warning: seed 'Activity' not found in index — skipped
GraphQL subgraph : 75 nodes
SDO subgraph     : 104 nodes


In [15]:
# what did the walker pull in?
describe(gql_subgraph, label="GraphQL")


════════════════════════════════════════════════════════════
  GraphQL subgraph  —  75 nodes
════════════════════════════════════════════════════════════

By node type:
  attribute          (  1)  UUID
  concept            ( 53)  ActivityMetadataFieldValue, ActivityStructureElement, Calendar, CalendarOperation, ConnectedPeriod, Data...
  enum               ( 19)  ActivityType, DayOfWeek, ImportStatus, OperationType, PlanningLevel, PlanningObjectTypes...
  seed               (  2)  Activity, Schedule

By hop distance from seeds:
  step 0  (  2 nodes)  Activity, Schedule
  step 1  ( 17 nodes)  ActivityMetadataFieldValue, ActivityType, Calendar, Data, MetadataFieldValue, ReportActivity, ReportActivityArchive, ReportBaselineRevision...
  step 2  ( 34 nodes)  ActivityStructureElement, CalendarOperation, IlapFile, ImportStatus, MetadataField, PointsInTimeType, Profile, ReportActivityMetadataFieldValue...
  step 3  ( 15 nodes)  ConnectedPeriod, IlapFileProgress, OperationType, PlanningLevel,

In [16]:
describe(sdo_subgraph, label="SDO")


════════════════════════════════════════════════════════════
  SDO subgraph  —  104 nodes
════════════════════════════════════════════════════════════

By node type:
  attribute          ( 24)  activityCancellationDate, actualFinishDateTime, actualStartDateTime, earlyFinishDateTime, earlyStartDateTime, effortPersonHours...
  concept            ( 47)  ActivityBreakdownStructure, ActualEffortDatum, AlwaysOnScheduleActivity, BaselineSchedule, CancelledActivity, ContinuousWorkPatternAssignment...
  relation           ( 31)  actualFinish, actualStart, earlyFinish, earlyStart, executedOnSite, finishNoEarlierThan...
  seed               (  2)  Project, Schedule

By hop distance from seeds:
  step 0  (  2 nodes)  Project, Schedule
  step 1  ( 10 nodes)  BaselineSchedule, LiveSchedule, effortPersonHours, hasEffort, hasPlannedDuration, hasProgress, hasSchedule, hasSchedulePart...
  step 2  (  9 nodes)  EffortDatum, Milestone, PlannedDuration, ProgressDatum, ScheduleActivity, ScheduleActivityLin

## 3 — understand rapidfuzz before running the full match

Try a few pairs manually to understand what the scores mean.

In [ ]:
from rapidfuzz import fuzz

pairs = [
    # identical concept, same name
    ("earlyStart",           "earlyStart"),
    # identical concept, one has DateTime suffix
    ("earlyStart",           "earlyStartDateTime"),
    # same concept, different naming convention
    ("cancelledDate",        "activityCancellationDate"),
    # same concept, completely different names
    ("plannedWorkHours",     "hasPlannedEffort"),
    # word order difference
    ("startNoEarlierThan",   "noEarlierThanStart"),
    # unrelated
    ("earlyStart",           "lagHours"),
]

print(f"{'GraphQL':<30} {'SDO':<30} {'ratio':>6} {'partial':>8} {'token_sort':>11}")
print("-" * 80)
for a, b in pairs:
    print(
        f"{a:<30} {b:<30}"
        f"{fuzz.ratio(a.lower(), b.lower()):>6.0f}"
        f"{fuzz.partial_ratio(a.lower(), b.lower()):>8.0f}"
        f"{fuzz.token_sort_ratio(a.lower(), b.lower()):>11.0f}"
    )

## 4 — run the full match

In [ ]:
from etl.matcher import match, summary

results = match(gql_subgraph, sdo_subgraph)
summary(results)

## 5 — explore results as a DataFrame

In [ ]:
from etl.matcher import to_dataframe

df = to_dataframe(results)
print(f"Total rows: {len(df)}")
df.head(10)

In [ ]:
# how many in each bucket?
df["bucket"].value_counts()

In [ ]:
# identical matches only — sorted by score
df[df["bucket"] == "identical"].sort_values("score", ascending=False)

In [ ]:
# equivalent matches — these need SME review
df[df["bucket"] == "equivalent"].sort_values("score", ascending=False)

In [ ]:
# unique to GraphQL — no SDO counterpart found
df[df["bucket"] == "unique_gql"]

In [ ]:
# unique to SDO — no GraphQL counterpart found
df[df["bucket"] == "unique_sdo"]

## 6 — adjust thresholds and rerun

The default thresholds are `identical >= 92` and `equivalent >= 60`.
Try different values and see how the bucket sizes change.

In [ ]:
results_strict = match(
    gql_subgraph,
    sdo_subgraph,
    threshold_identical=95,
    threshold_equivalent=70,
)
summary(results_strict)

## 7 — export to Excel for SME review

In [ ]:
output_path = ROOT / "mapping_candidates.xlsx"

df.to_excel(output_path, index=False)
print(f"Saved to {output_path}")

## 8 — what comes next

Rapidfuzz catches lexical matches — same or similar names.
It misses conceptual matches where the names are completely different:

```
plannedWorkHours  ↔  hasPlannedEffort     → low lexical score, but same concept
cancelledDate     ↔  activityCancellationDate  → medium score, same concept
```

The next step is `sentence-transformers` — semantic embedding matching.
It embeds `field name + description` as a vector and compares meaning,
not spelling. That will re-examine the `equivalent` and `unique_gql` buckets
and surface matches that rapidfuzz missed.